# Ecolocalización

**El Sexto Sentido del Murciélago 🦇**

¿Alguna vez te has preguntado cómo los murciélagos pueden volar en la oscuridad total sin chocar con las ramas de los árboles? En este laboratorio, transformaremos nuestro Ladrillo NXT en una criatura capaz de "ver" usando el sonido.



<div style="text-align: center; margin-bottom: 20px;">
    <video width="100%" style="max-width: 800px; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.1);" controls>
        <source src="_static/videos/actividad_1.mp4" type="video/mp4">
        Tu navegador no soporta la reproducción de video.
    </video>
</div>


In [1]:
import random
from IPython.display import display, HTML

# UID para evitar colisiones en Jupyter Book cuando hay múltiples simulaciones
uid = str(random.randint(10000, 99999))

simulacion_html = """
<div style="border: 1px solid #e0e0e0; padding: 20px; border-radius: 8px; background-color: #f8f9fa; font-family: -apple-system, sans-serif;">
    <h3 style="margin-top:0; color: #2c3e50;">Simulación: Ecolocalización del Robot (Sensor Ultrasónico)</h3>
    <p style="font-size: 0.95em; color: #555; margin-bottom: 15px;">
        Presiona <strong>Reproducir</strong>. Observa y escucha cómo el tiempo de espera entre cada "Beep" disminuye de forma proporcional a la distancia, tal como está programado en NXC: <code>Wait(distancia * 10);</code>. El robot debe detenerse a 10 cm del muro.
    </p>
    
    <div style="display: flex; gap: 20px; margin-bottom: 20px; background: #e9ecef; padding: 15px; border-radius: 8px; flex-wrap: wrap; align-items: center; justify-content: space-between;">
        <div style="display: flex; gap: 15px; align-items: center;">
            <button id="play_UID" style="background-color: #198754; color: white; border: none; padding: 10px 20px; border-radius: 5px; cursor: pointer; font-weight: bold; font-size: 14px;">▶ Reproducir</button>
            <button id="reset_UID" style="background-color: #6c757d; color: white; border: none; padding: 10px 20px; border-radius: 5px; cursor: pointer; font-weight: bold; font-size: 14px;">🔄 Reiniciar</button>
        </div>
        
        <div style="display: flex; gap: 20px; align-items: center;">
            <div style="display: flex; flex-direction: column; align-items: center; background: #fff; padding: 8px 15px; border-radius: 5px; border: 1px solid #ccc; width: 140px;">
                <label style="font-weight: 600; font-size: 0.8em; margin-bottom: 2px; color: #0d6efd;">Distancia (Sensor)</label>
                <span id="dist_val_UID" style="font-weight: bold; font-family: monospace; font-size: 1.1em; color: #212529;">40.0 cm</span>
            </div>
            <div style="display: flex; flex-direction: column; align-items: center; background: #fff; padding: 8px 15px; border-radius: 5px; border: 1px solid #ccc; width: 140px;">
                <label style="font-weight: 600; font-size: 0.8em; margin-bottom: 2px; color: #dc3545;">Tiempo (Wait)</label>
                <span id="wait_val_UID" style="font-weight: bold; font-family: monospace; font-size: 1.1em; color: #212529;">400 ms</span>
            </div>
        </div>
    </div>
    
    <div style="display: flex; justify-content: center; margin-bottom: 10px;">
        <canvas id="canvas_sim_UID" width="600" height="200" style="background: #ffffff; border: 1px solid #ccc; border-radius: 4px; box-shadow: 0 2px 5px rgba(0,0,0,0.05); max-width: 100%;"></canvas>
    </div>
    
    <div style="text-align: center; font-size: 0.85em; color: #6c757d; margin-top: 5px;">
        <i>Nota: Para escuchar los tonos (440Hz y 880Hz), asegúrate de tener el volumen encendido.</i>
    </div>
</div>

<script>
    (function() {
        function initSimulation() {
            const canvas = document.getElementById('canvas_sim_UID');
            if (!canvas) {
                setTimeout(initSimulation, 50); 
                return;
            }
            
            const ctx = canvas.getContext('2d');
            const btn_play = document.getElementById('play_UID');
            const btn_reset = document.getElementById('reset_UID');
            const lbl_dist = document.getElementById('dist_val_UID');
            const lbl_wait = document.getElementById('wait_val_UID');
            
            // Audio Context para generar los Beeps como el NXT
            let audioCtx = null;
            
            // Variables Físicas y de Simulación
            let isPlaying = false;
            let animationId;
            let lastTime = 0;
            
            // Escala: 10 pixeles = 1 cm
            const PX_PER_CM = 10;
            const Y_FLOOR = 150;
            const WALL_X = 550; // Posición de la pared
            const START_X = 50; // Posición inicial del robot
            const UMBRAL_CM = 10; // Detenerse a 10 cm
            
            let robotX = START_X;
            let robotSpeed = 60; // Pixeles por segundo
            
            // Lógica del "Wait" y el Sonido
            let lastBeepTime = 0;
            let isBeepingVisual = false; 
            let visualBeepTimer = 0;
            let hasStopped = false; // Bandera para el tono final largo
            
            function initAudio() {
                if (!audioCtx) {
                    audioCtx = new (window.AudioContext || window.webkitAudioContext)();
                }
                if (audioCtx.state === 'suspended') {
                    audioCtx.resume();
                }
            }
            
            function playTone(frequency, durationMs) {
                if (!audioCtx) return;
                const osc = audioCtx.createOscillator();
                const gain = audioCtx.createGain();
                
                osc.type = 'square'; // Sonido similar al chip del NXT
                osc.frequency.value = frequency;
                
                osc.connect(gain);
                gain.connect(audioCtx.destination);
                
                osc.start();
                // Fade out suave para evitar clicks en los parlantes
                gain.gain.setValueAtTime(0.1, audioCtx.currentTime);
                gain.gain.exponentialRampToValueAtTime(0.001, audioCtx.currentTime + (durationMs / 1000));
                osc.stop(audioCtx.currentTime + (durationMs / 1000));
                
                // Efecto visual
                isBeepingVisual = true;
                visualBeepTimer = durationMs / 1000;
            }

            function drawRobot(x, y) {
                // Ruedas
                ctx.fillStyle = "#333";
                ctx.fillRect(x + 5, y - 10, 15, 20); // Rueda trasera
                ctx.fillRect(x + 40, y - 10, 15, 20); // Rueda delantera
                
                // Ladrillo NXT / Chasis
                ctx.fillStyle = "#f8f9fa";
                ctx.strokeStyle = "#adb5bd";
                ctx.lineWidth = 2;
                ctx.beginPath();
                ctx.roundRect(x, y - 40, 60, 35, 5);
                ctx.fill();
                ctx.stroke();
                
                // Pantalla NXT
                ctx.fillStyle = "#c2d1b2"; // Color verde retro
                ctx.fillRect(x + 10, y - 35, 20, 15);
                
                // Sensor Ultrasónico (Ojos)
                ctx.fillStyle = "#ffc107";
                ctx.beginPath();
                ctx.arc(x + 60, y - 20, 8, -Math.PI/2, Math.PI/2);
                ctx.fill();
                ctx.fillStyle = "#212529";
                ctx.beginPath();
                ctx.arc(x + 60, y - 25, 3, 0, Math.PI*2);
                ctx.arc(x + 60, y - 15, 3, 0, Math.PI*2);
                ctx.fill();
                
                // Indicador visual de Beep
                if (isBeepingVisual) {
                    ctx.fillStyle = "#dc3545";
                    ctx.font = "bold 14px sans-serif";
                    ctx.fillText("BEEP!", x + 15, y - 50);
                    
                    // Ondas de sonido amarillas
                    ctx.strokeStyle = "#ffc107";
                    ctx.lineWidth = 3;
                    ctx.beginPath();
                    ctx.arc(x + 70, y - 20, 15, -Math.PI/4, Math.PI/4);
                    ctx.stroke();
                    ctx.beginPath();
                    ctx.arc(x + 70, y - 20, 25, -Math.PI/4, Math.PI/4);
                    ctx.stroke();
                }
            }

            function drawSim() {
                ctx.clearRect(0, 0, canvas.width, canvas.height);
                
                // Dibujar Piso
                ctx.fillStyle = "#e9ecef";
                ctx.fillRect(0, Y_FLOOR, canvas.width, canvas.height - Y_FLOOR);
                ctx.strokeStyle = "#ced4da";
                ctx.beginPath(); ctx.moveTo(0, Y_FLOOR); ctx.lineTo(canvas.width, Y_FLOOR); ctx.stroke();
                
                // Dibujar Pared
                ctx.fillStyle = "#6c757d";
                ctx.fillRect(WALL_X, 20, 30, Y_FLOOR - 20);
                // Patrón de ladrillos en la pared
                ctx.strokeStyle = "#495057";
                ctx.lineWidth = 1;
                for(let i=30; i<Y_FLOOR; i+=15) {
                    ctx.beginPath(); ctx.moveTo(WALL_X, i); ctx.lineTo(WALL_X + 30, i); ctx.stroke();
                }
                
                // Calcular distancia actual
                let distancePx = WALL_X - (robotX + 60); // 60 es el frente del robot
                let distanceCm = Math.max(0, distancePx / PX_PER_CM);
                let waitTimeMs = distanceCm * 10;
                
                // Actualizar Etiquetas UI
                lbl_dist.innerText = distanceCm.toFixed(1) + " cm";
                if (!hasStopped) {
                    lbl_wait.innerText = Math.round(waitTimeMs) + " ms";
                } else {
                    lbl_wait.innerText = "Detenido";
                    lbl_wait.style.color = "#dc3545";
                }
                
                // Dibujar línea de medición
                ctx.strokeStyle = "#0d6efd";
                ctx.setLineDash([4, 4]);
                ctx.beginPath();
                ctx.moveTo(robotX + 60, Y_FLOOR + 25);
                ctx.lineTo(WALL_X, Y_FLOOR + 25);
                ctx.stroke();
                ctx.setLineDash([]);
                
                // Textos
                ctx.fillStyle = "#0d6efd";
                ctx.font = "12px sans-serif";
                ctx.fillText("d = " + distanceCm.toFixed(1) + " cm", robotX + 60 + (distancePx/2) - 30, Y_FLOOR + 40);
                
                ctx.fillStyle = "#dc3545";
                ctx.fillText("Umbral: 10 cm", WALL_X - 90, 15);
                ctx.strokeStyle = "#dc3545";
                ctx.setLineDash([2, 2]);
                ctx.beginPath();
                ctx.moveTo(WALL_X - (UMBRAL_CM * PX_PER_CM), 20);
                ctx.lineTo(WALL_X - (UMBRAL_CM * PX_PER_CM), Y_FLOOR);
                ctx.stroke();
                ctx.setLineDash([]);
                
                drawRobot(robotX, Y_FLOOR);
            }

            function animate(timestamp) {
                if (!lastTime) lastTime = timestamp;
                let deltaTime = (timestamp - lastTime) / 1000;
                lastTime = timestamp;
                
                if (isPlaying) {
                    let distancePx = WALL_X - (robotX + 60);
                    let distanceCm = distancePx / PX_PER_CM;
                    let waitTimeMs = distanceCm * 10;
                    
                    if (distanceCm > UMBRAL_CM) {
                        // El robot avanza
                        robotX += robotSpeed * deltaTime;
                        
                        // Lógica del "Wait" de NXC
                        if (timestamp - lastBeepTime > waitTimeMs) {
                            playTone(440, 50); // Tono de 440Hz por 50ms
                            lastBeepTime = timestamp;
                        }
                    } else {
                        // Condición else (Detenerse)
                        if (!hasStopped) {
                            playTone(880, 1000); // Tono agudo de alerta por 1s
                            hasStopped = true;
                            isPlaying = false;
                            btn_play.innerText = "▶ Reproducir";
                            btn_play.style.backgroundColor = "#198754";
                        }
                    }
                }
                
                // Temporizador visual del beep
                if (isBeepingVisual) {
                    visualBeepTimer -= deltaTime;
                    if (visualBeepTimer <= 0) isBeepingVisual = false;
                }
                
                drawSim();
                
                if(isPlaying || isBeepingVisual) {
                    animationId = requestAnimationFrame(animate);
                }
            }

            btn_play.addEventListener('click', () => {
                initAudio();
                if (hasStopped) return; // Si ya llegó al muro, no hacer nada
                
                isPlaying = !isPlaying;
                if (isPlaying) {
                    btn_play.innerText = "⏸ Pausar";
                    btn_play.style.backgroundColor = "#ffc107";
                    btn_play.style.color = "#000";
                    lastTime = 0; 
                    animationId = requestAnimationFrame(animate);
                } else {
                    btn_play.innerText = "▶ Reproducir";
                    btn_play.style.backgroundColor = "#198754";
                    btn_play.style.color = "#fff";
                }
            });

            btn_reset.addEventListener('click', () => {
                isPlaying = false;
                hasStopped = false;
                robotX = START_X;
                lastBeepTime = 0;
                isBeepingVisual = false;
                lbl_wait.style.color = "#212529";
                
                btn_play.innerText = "▶ Reproducir";
                btn_play.style.backgroundColor = "#198754";
                btn_play.style.color = "#fff";
                
                if (audioCtx) {
                    // Detener cualquier sonido remanente
                    audioCtx.suspend();
                    setTimeout(() => audioCtx.resume(), 100);
                }
                
                drawSim();
            });
            
            // Dibujar estado inicial
            drawSim();
        }
        
        initSimulation();
    })();
</script>
"""

html_final = simulacion_html.replace('UID', uid)
display(HTML(html_final))


## 1. Guía del Estudiante

### El Desafío
Tu misión es construir y programar un robot que avance en línea recta hacia un muro. A medida que se acerque, el robot debe emitir un sonido que se acelere, simulando la **ecolocalización** de un murciélago o un delfín. Finalmente, debe detenerse automáticamente a exactamente 10 cm del muro para evitar el choque.

### Fase 1: Ingeniería (Construcción)
1. Arma la **Base Motriz Estándar** conectando los motores a los **puertos B y C**.
2. Fija el **Sensor Ultrasónico** en la parte frontal del robot, apuntando hacia adelante.
3. Conecta el cable del sensor ultrasónico al **Puerto 4** del Ladrillo Inteligente.

### Fase 2: Programación (Código NXC)
El murciélago necesita un cerebro. Abre **Bricx Command Center**, crea un nuevo archivo y transcribe el siguiente código. Lee los comentarios (textos después de `//`) para entender qué hace cada línea.

```c
// Definimos a qué distancia queremos que el robot se detenga (en centímetros)
#define UMBRAL 10 

task main() {
    // Le indicamos al robot que hay un sensor ultrasónico en el Puerto 4
    SetSensorLowspeed(S4);
    
    // Creamos una variable para guardar la distancia que el sensor lee
    int distancia;

    while(true) {
        // Leemos el sensor y guardamos el valor
        distancia = SensorUS(S4);

        if (distancia > UMBRAL) {
            // Si la distancia es mayor a 10 cm:
            OnFwd(OUT_BC, 50);         // Avanzamos al 50% de potencia
            PlayTone(440, 50);         // Emitimos un tono corto (beep) de 50 milisegundos
            
            // LA MAGIA DE LA MATEMÁTICA: 
            // Esperamos un tiempo proporcional a la distancia antes del siguiente beep.
            // Si está lejos (ej. 50cm), espera 500ms. Si está cerca (15cm), espera 150ms.
            Wait(distancia * 10);      
            
        } else {
            // Si la distancia es 10 cm o menor (¡Cuidado, muro!):
            Off(OUT_BC);               // Apagamos los motores de golpe
            PlayTone(880, 1000);       // Emitimos un tono agudo y largo de 1 segundo
            Wait(1000);                // Esperamos que termine el sonido
            break;                     // Terminamos el programa
        }
    }
}
```


### Fase 3: Pruebas y Reflexión
* **Compila y descarga** el código al NXT.
* Coloca el robot en el suelo a unos 2 metros de una pared y ejecútalo.
* **Responde en tu cuaderno:** ¿Qué sucede con el sonido si en lugar de una pared plana, pones una caja en un ángulo inclinado frente al robot? ¿Por qué crees que ocurre esto con las ondas de sonido?

---

## 2. Guía del Docente

> **Nota para el Profesor:** Esta actividad está diseñada para ocultar la complejidad de la programación dentro de un experimento de biología y física aplicada. Evite centrarse únicamente en la sintaxis y priorice el análisis del comportamiento del robot.

### Vínculo Curricular (MINEDUC Chile)
* **Ciencias Naturales (6º Básico):** OA 1 - Reconocer y explicar que los seres vivos están formados por una o más células y que estas se organizan en tejidos, órganos y sistemas (vinculable al sistema nervioso y adaptación sensorial de los animales).
* **Tecnología (7º Básico):** OA 2 - Diseñar, hacer y probar objetos o sistemas tecnológicos frente a un problema.
* **Física (1º Medio):** OA 4 - Explicar fenómenos del sonido (reflexión, eco) utilizando el modelo ondulatorio.

### Conceptos Clave a Desarrollar
1. **Ecolocalización y Eco (Física):** El sensor ultrasónico emite una onda de sonido de alta frecuencia (inaudible para el ser humano). La onda viaja, rebota en el obstáculo y vuelve al sensor. El microprocesador mide el tiempo que tardó la onda en ir y volver, y usando la velocidad del sonido, calcula la distancia.
2. **Control Estructural (Programación):** Uso de la estructura cíclica `while(true)` (ciclo infinito) y la bifurcación condicional `if/else`.
3. **Variables Dinámicas:** La línea `Wait(distancia * 10);` es el corazón de la clase. Demuestra cómo una variable física (distancia) altera dinámicamente el comportamiento del sistema (tiempo de espera).

### Resolución de Problemas (Troubleshooting)
* **El robot choca con la pared:** Verifique que el cable del sensor esté firmemente conectado al Puerto 4 y los motores en los puertos B y C. Confirme que el sensor esté perfectamente horizontal.
* **Lecturas erráticas (el robot se detiene de la nada):** El sensor ultrasónico tiene un campo de visión cónico. Si está montado muy cerca del suelo, el eco puede estar rebotando en el piso. Pida a los alumnos que instalen el sensor un poco más arriba en la base motriz.
* **La prueba de la inclinación:** Si la pared está en ángulo (como a 45 grados), la onda de sonido "rebota" y se desvía, por lo que el eco nunca vuelve al sensor y el robot chocará irremediablemente. Esta es una excelente oportunidad para explicar cómo funcionan los aviones furtivos (Stealth) para evadir los radares militares.